In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Geometry based Machine Learning Methods for Bioimage Classification

## Introduction

Bioimage analysis is fundamental in understanding cellular structures and functions. By extracting meaningful features from bioimages, researchers can classify, segment, and recognize patterns within various cellular components. These features are broadly categorized into texture features and geometrical features.

- Texture Features: Capture the spatial arrangement and intensity distribution of pixels.
- Geometrical Features: Describe the shape and structure of objects within the image.

This notebook delves into:
- Histogram Analysis: Extracting first-order statistical features.
- Gray Level Co-occurrence Matrix (GLCM): Deriving second-order texture features.
- Haralick Texture Features: Comprehensive texture analysis leveraging GLCM.
- Geometrical Features: Characterizing the shape and structure of objects.
- Dimensionality Reduction and Visualization: Using UMAP and t-SNE to visualize high-dimensional feature spaces.

## Outline

1. Introduction
2. About the Hela Dataset
3. Setup and Libraries
4. Dataset Overview
5. Understanding the Dataset Statistics
6. Data Loading and Preprocessing
7. Label Encoding
8. Sample Images
9. Exploratory Data Analysis (EDA)
10. Feature Extraction Techniques:
    *     Histogram Analysis
    *     Gray Level Co-occurrence Matrix (GLCM)
    *     Haralick Texture Features
    *     Geometrical Features
14. Feature Matrix Preparation
15. Dimensionality Reduction and Visualization
16. Comparative Analysis: UMAP vs. t-SNE
17. Classical Machine Learning Methods
19. Conclusion
20. Future Directions


## Setup and Libraries

Before diving into the analysis, it's essential to set up the environment by installing the necessary libraries.



In [ ]:
# Uncomment the following lines to install required libraries
# !pip install numpy pandas matplotlib seaborn scikit-image mahotas scikit-learn umap-learn


Mahotas is a computer vision and image processing library for Python.

It includes many algorithms implemented in C++ for speed while operating in numpy arrays and with a very clean Python interface.

In [ ]:
!pip install mahotas

In [ ]:
!pip install umap-learn

### Importing Required Libraries

Ensure that all required libraries are installed. You can install any missing libraries using `pip`. Uncomment and run the below cell if needed.

In [ ]:
# Importing essential libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Skimage for image processing
from skimage import io, color, feature, measure, morphology
from skimage.filters import threshold_otsu
from skimage.transform import resize
from scipy.stats import skew, kurtosis
from skimage.feature import graycomatrix, graycoprops
from skimage.measure import label, regionprops


# Mahotas for Haralick features
import mahotas

# Scipy for statistical computations
from scipy.stats import skew, kurtosis

# Scikit-learn for preprocessing, dimensionality reduction and ML models
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_curve, average_precision_score


# UMAP for dimensionality reduction
import umap

# Suppress FutureWarnings from seaborn
import warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

# Setting plot aesthetics
%matplotlib inline
sns.set(style='whitegrid', palette='muted', font_scale=1.2)

# Verify skimage version to ensure 'graycomatrix' is available
import skimage
print(f"Skimage version: {skimage.__version__}")



If you encounter the following error:

> ImportError: cannot import name 'graycomatrix' from 'skimage.feature'

It indicates that your skimage version is outdated. To resolve this, upgrade scikit-image using the following command:
> !pip install --upgrade scikit-image

# Dataset Overview
The Hela dataset is a collection of bioimages derived from HeLa cells, which are a widely used immortal cell line in scientific research.

The HeLa cell dataset involves cells grown on collagen-coated coverslips, fixed, and permeabilized, followed by labeling with antibodies targeting various organelle proteins or rhodamine phalloidin for actin. Fluorescence microscopy was used to collect images in three stacks for different channels (Cy5, Cy3, DAPI), focusing on distinct organelles and cellular structures.

It is organized into three main directories: train, test, and valid, each containing subdirectories corresponding to different classes. The classes include:

* ActinFilaments
* ER (Endoplasmic Reticulum)
* Endosome
* Golgi_gia (Golgi apparatus - Giant)
* Golgi_gpp (Golgi apparatus - GPP variant)
* Lysosome
* Microtubules
* Mitochondria
* Nucleolus
* Nucleus

Each subdirectory houses multiple images representing instances of the respective class.

## Step One: Understanding the Dataset Statistics

Let's examine the basic statistics of the dataset, including the number of images per class and their distribution across the training, testing, and validation splits. This analysis is essential to ensure that our feature extraction and subsequent analysis are based on a balanced and representative dataset.

The table below presents the distribution of images across different classes and dataset splits. A balanced distribution helps in preventing bias during feature extraction and model training, ensuring that each class is adequately represented.


In [ ]:
# Define the dataset path
DATASET_PATH = '/content/drive/MyDrive/02740_data/recitation_4/Hela'  # Replace with your actual dataset path

# Verify the dataset path exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"The dataset path {DATASET_PATH} does not exist. Please check the path.")

# Function to count images per class and split
def dataset_statistics(dataset_path):
    stats = {}
    for split in ['train', 'test', 'valid']:
        split_path = os.path.join(dataset_path, split)
        if not os.path.isdir(split_path):
            print(f"Warning: {split_path} does not exist.")
            continue
        stats[split] = {}
        for label in os.listdir(split_path):
            label_path = os.path.join(split_path, label)
            if not os.path.isdir(label_path):
                continue
            count = len([file for file in os.listdir(label_path) if file.endswith('.png')])
            stats[split][label] = count
    return stats

# Get dataset statistics
stats = dataset_statistics(DATASET_PATH)

# Convert to DataFrame for better visualization
stats_df = pd.DataFrame(stats).fillna(0).astype(int)
stats_df = stats_df.transpose()
stats_df


## Step Two: Data Loading and Preprocessing

In this section, we'll load the images from the dataset, preprocess them, and assign labels. Preprocessing steps are crucial for standardizing the data and preparing it for feature extraction.

* Grayscale Conversion: All images are converted to grayscale to simplify the analysis and reduce computational complexity. Grayscale images retain essential structural information while eliminating color variations that may not be relevant for texture and geometrical feature extraction.

* Resizing: Images are resized to a uniform dimension. Standardizing image sizes ensures consistency across all samples, which is vital for reliable feature extraction and comparison.


In [ ]:
# Modified load_data function to include split information
def load_data_with_splits(dataset_path, image_size=(256, 256)):
    data = []
    labels = []
    splits = []
    split_names = ['train', 'test', 'valid']
    for split in split_names:
        split_path = os.path.join(dataset_path, split)
        if not os.path.isdir(split_path):
            print(f"Warning: {split_path} does not exist.")
            continue
        for label in os.listdir(split_path):
            label_path = os.path.join(split_path, label)
            if not os.path.isdir(label_path):
                continue
            for file in os.listdir(label_path):
                if file.endswith('.png'):
                    file_path = os.path.join(label_path, file)
                    try:
                        # Read the image in grayscale
                        image = io.imread(file_path, as_gray=True)
                        # Resize the image
                        image_resized = resize(image, image_size, anti_aliasing=True)
                        # Normalize the image
                        image_normalized = image_resized / np.max(image_resized)
                        data.append(image_normalized)
                        labels.append(label)
                        splits.append(split)
                    except Exception as e:
                        print(f"Error loading {file_path}: {e}")
    return np.array(data), np.array(labels), np.array(splits)

# Load the data with split information
images, labels, split_labels = load_data_with_splits(DATASET_PATH)

print(f'Loaded {len(images)} images with {len(np.unique(labels))} unique classes.')


## Step Three: Label Encoding

Machine learning algorithms require numerical input rather than categorical labels. Therefore, we'll encode the string labels into numerical values using label encoding. This process transforms each class label into a unique integer, facilitating its use in computational models.

Understanding label encoding is essential for interpreting the results of classification tasks and visualizations. Each class label is mapped to a unique integer, ensuring that the encoding process maintains a one-to-one correspondence between classes and numerical values.


In [ ]:
# Initialize LabelEncoder
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

# Mapping of labels to encoded values
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label Mapping:")
print(label_mapping)

## Step Four: Sample Images

To gain a better understanding of the dataset, we'll visualize some sample images from each class. Visual inspection of sample images helps in assessing the diversity and complexity of the data, providing intuitive insights into the characteristics of each class.


In [ ]:
# Function to display sample images per class with class labels below the images
def display_sample_images(images, labels, num_samples=5):
    unique_labels = np.unique(labels)
    num_classes = len(unique_labels)
    plt.figure(figsize=(num_samples * 3, num_classes * 3))

    for idx, label in enumerate(unique_labels):
        # Select images corresponding to the current class label
        label_images = images[labels == label]
        # Pick the specified number of samples
        selected_images = label_images[:num_samples]

        for i, img in enumerate(selected_images):
            ax = plt.subplot(num_classes, num_samples, idx * num_samples + i + 1)
            plt.imshow(img, cmap='gray')
            plt.axis('off')
            # Display the class label below each image
            ax.set_title(f'{label}', fontsize=10, pad=5)

    plt.suptitle('Sample Images from Each Class', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

# Display sample images
display_sample_images(images, labels, num_samples=5)


## Step Five: Exploratory Data Analysis (EDA)

Before diving into feature extraction, it's beneficial to perform Exploratory Data Analysis (EDA) to understand the dataset's characteristics better. EDA involves visualizing and summarizing the data to uncover patterns, anomalies, and insights that inform subsequent analysis.

### Class Distribution

We'll visualize the distribution of images across different classes and dataset splits. Understanding class distribution is crucial for identifying any imbalances that may affect feature extraction and model training.


In [ ]:
# Function to plot class distribution
def plot_class_distribution(stats_df):
    stats_df.plot(kind='bar', figsize=(15,7))
    plt.title('Number of Images per Class in Each Split')
    plt.xlabel('Dataset Split')
    plt.ylabel('Number of Images')
    plt.legend(title='Classes', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# Plot class distribution
plot_class_distribution(stats_df)


### Image Intensity Distribution

Analyzing the intensity distribution of the images provides insights into the variability and quality of the dataset. Examining pixel intensity histograms helps in understanding contrast and brightness variations within the images, which are important for texture analysis.


In [ ]:
# Plot intensity distribution for a sample of images
def plot_intensity_distribution(images, num_samples=5):
    plt.figure(figsize=(15,10))
    for i in range(num_samples):
        img = images[i]
        plt.subplot(num_samples, 2, 2*i+1)
        plt.imshow(img, cmap='gray')
        plt.title(f'Image {i+1}')
        plt.axis('off')
        plt.subplot(num_samples, 2, 2*i+2)
        sns.histplot(img.flatten(), bins=50, kde=True)
        plt.title(f'Intensity Histogram {i+1}')
    plt.tight_layout()
    plt.show()

# Plot intensity distribution for first 5 images
plot_intensity_distribution(images, num_samples=5)


# Feature Extraction Techniques
Feature extraction is a critical step in image analysis that involves deriving quantitative information from images to represent their characteristics numerically. In the context of bioimage analysis, these features can describe various aspects of the images, such as intensity, texture, and shape, which are essential for distinguishing between different classes.

We employ the following feature extraction techniques:

* Histogram Analysis: This technique extracts first-order statistical features from the pixel intensity distribution within the image. For bioimages, these features—such as Mean, Standard Deviation, Skewness, Kurtosis, and Entropy—provide essential information about the overall intensity patterns, helping to identify variations in brightness, contrast, and complexity within cellular or subcellular structures. These features are particularly useful for distinguishing different cell types or identifying abnormalities in tissue samples.

* Gray Level Co-occurrence Matrix (GLCM): GLCM captures second-order texture information by examining the spatial relationships between pixel intensities. In bioimages, this technique helps quantify texture patterns that are not immediately visible to the naked eye, such as the arrangement of fibrous structures, granularity, or tissue homogeneity. By analyzing features like Contrast, Correlation, Energy, and Homogeneity, GLCM provides insights into how textures vary across different biological samples, making it useful for identifying structural anomalies in cellular and tissue images.

* Haralick Texture Features: Derived from the GLCM, Haralick features offer a comprehensive analysis of image textures by evaluating different texture properties like Angular Second Moment, Contrast, and Entropy. In bioimage analysis, these features help quantify the complexity and uniformity of textures, which can be crucial for tasks such as identifying cancerous cells, assessing fibrosis in tissues, or differentiating between healthy and diseased states. Using the mahotas library, we can efficiently extract these features to provide a detailed texture analysis.

* Geometrical Features: Geometrical features focus on the shape and structure of objects within bioimages, such as cells, nuclei, or other subcellular components. By analyzing features like Perimeter, Area, Compactness, Circularity, Eccentricity, and Solidity, this approach provides valuable shape descriptors that help differentiate objects based on their morphology. For example, distinguishing between round and elongated cells or measuring the compactness of a tissue region can be critical for diagnosing various conditions. These features are particularly useful in tasks involving segmentation and morphological classification of cells or tissue regions.

## Step Six: Histogram Analysis

Histogram analysis involves extracting first-order statistical features that describe the distribution of pixel intensities within an image. These features provide foundational insights into the image's intensity characteristics, which are pivotal for distinguishing between different classes.

**Features Extracted:**

- **Mean**: The average intensity value of the image.
- **Standard Deviation (StdDev)**: Measures the spread of intensity values around the mean.
- **Skewness**: Indicates the asymmetry of the intensity distribution.
- **Kurtosis**: Measures the "tailedness" of the distribution.
- **Entropy**: Quantifies the randomness or complexity in the intensity distribution.

The `extract_histogram_features` function processes each image to compute these statistical features by flattening the image into a one-dimensional array. This simplification treats the pixel intensities as a single distribution, facilitating the computation of these metrics.


In [ ]:
# Function to compute histogram-based features
def extract_histogram_features(image):
    # Flatten the image to a 1D array
    pixels = image.flatten()
    # Compute Mean
    mean = np.mean(pixels)
    # Compute Standard Deviation
    std = np.std(pixels)
    # Compute Skewness
    skewness = skew(pixels)
    # Compute Kurtosis
    kurt = kurtosis(pixels)
    # Compute Entropy
    # To compute entropy, ensure no zero values by adding a small constant
    # entropy = -np.sum(pixels * np.log2(pixels + 1e-9))
    image_quantized = (image * 7).astype(np.uint8)
    pixels = image_quantized.flatten()
    distribution = np.histogram(pixels, bins=8, range=(0, 8), density=True)[0]
    entropy = -np.sum(distribution * np.log2(distribution + 1e-9))
    return [mean, std, skewness, kurt, entropy]

# Extract histogram features for all images
hist_features = np.array([extract_histogram_features(img) for img in images])

# Create a DataFrame for histogram features
hist_df = pd.DataFrame(hist_features, columns=['Mean', 'StdDev', 'Skewness', 'Kurtosis', 'Entropy'])
hist_df.head()


In [ ]:
hist_df['labels'] = labels

### Visualizing Histogram Features

The pairplot provides a comprehensive visualization of how each histogram feature relates to the others. Such insights aid in identifying correlations and potential redundancies among features, which can inform feature selection and model optimization.


In [ ]:
# Pairplot of histogram features to visualize relationships
sns.pairplot(hist_df, hue="labels")
plt.suptitle('Pairplot of Histogram Features', y=1.02)
plt.show()


### Feature Distribution Analysis

Analyzing the distribution of each feature individually helps in understanding their range and variability. Boxplots and stripplots offer visual summaries of the feature distributions, highlighting aspects like median values, quartiles, and potential outliers. This analysis is instrumental in identifying patterns and ensuring that features contribute meaningfully to the analysis.


In [ ]:
# Boxplots for histogram features
plt.figure(figsize=(12,6))
sns.boxplot(data=hist_df)
plt.title('Boxplot of Histogram Features')
plt.show()

# Stripplots for granular distribution view to avoid overcrowding
plt.figure(figsize=(12,6))
sns.stripplot(data=hist_df, jitter=True, color=".25", size=3)
plt.title('Stripplot of Histogram Features')
plt.show()


## Step Seven: Gray Level Co-occurrence Matrix (GLCM)

The Gray Level Co-occurrence Matrix (GLCM) is a second-order statistical method that captures the spatial relationship between pixel intensities. By analyzing how frequently pairs of pixel values occur in a specified spatial relationship, GLCM provides rich texture information about the image.

### GLCM Concepts

- **Distances**: The separation distance between pixel pairs.
- **Angles**: The orientation between pixel pairs (e.g., 0°, 45°, 90°, 135°).
- **Symmetric Matrix**: Ensures that the GLCM is symmetric, capturing bidirectional relationships.

### Extracting GLCM Features

We will extract the following GLCM features:

- **Contrast**: Measures the intensity contrast between a pixel and its neighbor.
- **Correlation**: Measures how correlated a pixel is to its neighbor over the whole image.
- **Energy**: Also known as Angular Second Moment, it measures textural uniformity.
- **Homogeneity**: Measures the closeness of the distribution of elements in the GLCM to the GLCM diagonal.

Quantizing the image to 8 gray levels simplifies the GLCM computation without significantly compromising the texture information. The `greycomatrix` function generates the GLCM for the specified distances and angles, and `greycoprops` extracts the desired statistical properties.


In [ ]:
# Function to compute GLCM features
def extract_glcm_features(image, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4]):
    # Quantize the image to 8 gray levels to reduce computational complexity
    image_quantized = (image * 7).astype(np.uint8)
    # Compute GLCM
    glcm = graycomatrix(image_quantized, distances=distances, angles=angles, symmetric=True, normed=True)
    # Extract GLCM properties
    contrast = graycoprops(glcm, 'contrast').mean()
    correlation = graycoprops(glcm, 'correlation').mean()
    energy = graycoprops(glcm, 'energy').mean()
    homogeneity = graycoprops(glcm, 'homogeneity').mean()
    return [contrast, correlation, energy, homogeneity]

# Extract GLCM features for all images
glcm_features = np.array([extract_glcm_features(img) for img in images])

# Create a DataFrame for GLCM features
glcm_df = pd.DataFrame(glcm_features, columns=['Contrast', 'Correlation', 'Energy', 'Homogeneity'])
glcm_df.head()


### Visualizing GLCM Features

Visual representations aid in understanding the distribution and variance of GLCM features across the dataset. Boxplots and stripplots provide insights into the spread and distribution of GLCM features, helping in identifying outliers and understanding the variability within each feature.


In [ ]:
# Boxplot of GLCM features
plt.figure(figsize=(10,6))
sns.boxplot(data=glcm_df)
plt.title('Boxplot of GLCM Features')
plt.show()

# Stripplot of GLCM features for granular insights
plt.figure(figsize=(10,6))
sns.stripplot(data=glcm_df, jitter=True, color=".25", size=3)
plt.title('Stripplot of GLCM Features')
plt.show()


## Step Eight: Haralick Texture Features

Haralick features are a set of texture descriptors derived from the GLCM, offering a more comprehensive analysis of image textures. Utilizing the `mahotas` library, we can efficiently extract these features.

### Haralick Features Overview

Haralick proposed 14 different features, but we will focus on the most commonly used ones:

- **Angular Second Moment (Energy)**: Measures textural uniformity.
- **Contrast**: Measures local variations.
- **Correlation**: Measures how correlated a pixel is to its neighbor.
- **Sum of Squares**: Measures variance.
- **Inverse Difference Moment (Homogeneity)**: Measures textural homogeneity.
- **Entropy**: Measures randomness.

### Extracting Haralick Features Using Mahotas

The `extract_haralick_features_mahotas` function leverages the `mahotas` library to compute Haralick features efficiently. By selecting the mean of the features across different angles, we obtain a consolidated set of descriptors that encapsulate the texture information of the image.


In [ ]:
# Function to compute Haralick features using mahotas
def extract_haralick_features_mahotas(image):
    # Convert image to uint8 format
    image_uint8 = (image * 255).astype(np.uint8)
    # Compute Haralick features
    haralick = mahotas.features.haralick(image_uint8, return_mean=True)
    # Extract specific Haralick features
    # Mahotas returns 13 features; we'll select the first 6 for simplicity
    return haralick[:6]

# Extract Haralick features for all images
haralick_features = np.array([extract_haralick_features_mahotas(img) for img in images])

# Create a DataFrame for Haralick features
haralick_df = pd.DataFrame(haralick_features, columns=[
    'Angular Second Moment', 'Contrast', 'Correlation',
    'Sum of Squares', 'Inverse Difference Moment', 'Entropy'
])
haralick_df.head()


### Comprehensive Haralick Features

Leveraging the `mahotas` library to extract all standard Haralick features ensures accurate and comprehensive texture analysis. By extracting all standard Haralick features, we capture various aspects of the image's texture, significantly enhancing the discriminative power of subsequent machine learning models.


In [ ]:
# Function to compute all standard Haralick features using mahotas
def extract_all_haralick_features_mahotas(image):
    # Convert image to uint8 format
    image_uint8 = (image * 255).astype(np.uint8)
    # Compute Haralick features
    haralick = mahotas.features.haralick(image_uint8, return_mean=True)
    # Mahotas returns 13 Haralick features
    feature_names = [
        'Angular Second Moment', 'Contrast', 'Correlation', 'Sum of Squares',
        'Inverse Difference Moment', 'Sum Average', 'Sum Variance', 'Sum Entropy',
        'Entropy', 'Difference Variance', 'Difference Entropy',
        'Information Measure of Correlation 1', 'Information Measure of Correlation 2'
    ]
    return haralick, feature_names

# Extract all Haralick features for all images
haralick_features_all = []
feature_names_all = []
for img in images:
    haralick, names = extract_all_haralick_features_mahotas(img)
    haralick_features_all.append(haralick)
    feature_names_all = names  # Same for all

haralick_features_all = np.array(haralick_features_all)

# Create a DataFrame for all Haralick features
haralick_all_df = pd.DataFrame(haralick_features_all, columns=feature_names_all)
haralick_all_df.head()


### Visualizing Haralick Features

The pairplot facilitates the visualization of relationships between different Haralick features, aiding in the identification of correlated features and potential redundancies. This analysis is crucial for refining feature sets and improving model performance.


In [ ]:
# Pairplot of Haralick features to visualize inter-feature relationships
haralick_df['labels'] = labels
sns.pairplot(haralick_df, hue = "labels")
plt.suptitle('Pairplot of Haralick Features', y=1.02)
plt.show()


## Step Nine: Geometrical Features

Geometrical features characterize the shape and structure of objects within bioimages. These features are instrumental in differentiating classes based on morphological attributes.

### Geometrical Features Overview

We will extract the following geometrical features:

- **Perimeter**: The total length around the object.
- **Area**: The number of pixels comprising the object.
- **Compactness**: Measures how closely the shape of an object approaches that of a perfect circle.
- **Circularity**: Another measure of how circular an object is.
- **Eccentricity**: Measures the deviation of the object's shape from being circular.
- **Major Axis Length**: The length of the major axis of the ellipse that has the same normalized second central moments as the region.
- **Minor Axis Length**: The length of the minor axis of the ellipse.
- **Solidity**: The ratio of the area of the object to the area of its convex hull.

### Extracting Geometrical Features

The `extract_geometrical_features` function processes each image to compute the specified geometrical features. By thresholding and labeling connected regions, we isolate objects of interest and analyze their morphological attributes. Removing small objects helps in reducing noise and focusing on relevant structures.


In [ ]:
# Function to compute geometrical features
def extract_geometrical_features(image):
    # Threshold the image using Otsu's method to create a binary image
    thresh = threshold_otsu(image)
    binary = image > thresh
    # Remove small objects to reduce noise
    binary = morphology.remove_small_objects(binary, min_size=64)
    # Label connected regions
    labeled = label(binary)
    regions = regionprops(labeled)

    if len(regions) == 0:
        # If no regions are found, return zeros for all features
        return [0]*8

    # Assuming the largest region corresponds to the object of interest
    region = max(regions, key=lambda r: r.area)

    # Extract geometrical features
    perimeter = region.perimeter
    area = region.area
    compactness = (perimeter**2) / (4 * np.pi * area) if area > 0 else 0
    circularity = (4 * np.pi * area) / (perimeter**2) if perimeter > 0 else 0
    eccentricity = region.eccentricity
    major_axis_length = region.major_axis_length
    minor_axis_length = region.minor_axis_length
    solidity = region.solidity

    return [perimeter, area, compactness, circularity, eccentricity, major_axis_length, minor_axis_length, solidity]

# Extract geometrical features for all images
geom_features = np.array([extract_geometrical_features(img) for img in images])

# Create a DataFrame for geometrical features
geom_df = pd.DataFrame(geom_features, columns=[
    'Perimeter', 'Area', 'Compactness', 'Circularity',
    'Eccentricity', 'Major Axis Length', 'Minor Axis Length', 'Solidity'
])
geom_df.head()


## Visualizing Geometrical Features
Visualizing geometrical features helps in understanding their distribution and potential overlaps between classes. Boxplots and stripplots offer insights into the central tendency and dispersion of geometrical features. They are instrumental in identifying patterns, overlaps, and potential feature importance for classification tasks.


In [ ]:
# Boxplot of Geometrical features
plt.figure(figsize=(14,8))
sns.boxplot(data=geom_df)
plt.title('Boxplot of Geometrical Features')
plt.show()

# Stripplot of Geometrical features for detailed distribution
plt.figure(figsize=(14,8))
sns.stripplot(data=geom_df, jitter=True, color=".25", size=3)
plt.title('Stripplot of Geometrical Features')
plt.show()


### Robust Geometrical Feature Extraction
To enhance the geometrical characterization, we include additional features such as Eccentricity, Major Axis Length, Minor Axis Length, and Solidity. These features provide a more detailed shape analysis, enabling finer distinctions between classes. Since the additional geometrical features have already been included in the extraction function above, let us verify the inclusion by checking the DataFrame.

The inclusion of additional geometrical features ensures a comprehensive shape analysis, capturing subtle morphological differences that can be crucial for distinguishing between similar classes.


In [ ]:
geom_df.head()


In [ ]:
if 'labels' in hist_df.columns:
  hist_df = hist_df.drop('labels', axis=1)

if 'labels' in glcm_df.columns:
  glcm_df = glcm_df.drop('labels', axis=1)

if 'labels' in haralick_df.columns:
  haralick_df = haralick_df.drop('labels', axis=1)



## Step Ten: Feature Matrix Preparation

Before performing dimensionality reduction, it's essential to prepare the feature matrix by combining all extracted features, removing duplicates, and handling missing values. This preparation ensures the data is clean and suitable for analysis.

### Combining All Features

We will concatenate the histogram, GLCM, Haralick, and geometrical features into a single feature matrix. Combining all extracted features into one DataFrame provides a comprehensive feature matrix, capturing both intensity distributions and morphological characteristics of the images.


In [ ]:
# Combine all feature DataFrames horizontally

features_df = pd.concat([hist_df, glcm_df, haralick_df, geom_df], axis=1)
features_df.head()


### Removing Duplicate Columns

Duplicate features can lead to redundancy and potentially skew the analysis. Removing duplicate columns ensures that each feature contributes uniquely to the analysis, enhancing the efficiency and accuracy of subsequent processing steps.


In [ ]:
# Check for duplicate columns
duplicate_columns = features_df.columns[features_df.columns.duplicated()]
print(f'Duplicate Columns: {duplicate_columns.tolist()}')

# Remove duplicate columns
features_df = features_df.loc[:, ~features_df.columns.duplicated()]
print(f'Feature matrix shape after removing duplicates: {features_df.shape}')


### Handling Missing Values

Missing values can adversely affect dimensionality reduction and machine learning models. We'll identify and impute missing values using the mean strategy. Upon inspection, there are no missing values in our feature matrix. However, including the imputation step ensures that our pipeline is robust and can handle datasets with missing values in future analyses.


In [ ]:
# Check for missing values in the feature matrix
missing_values = features_df.isnull().sum()
print(f'Missing values before imputation:\n{missing_values}')

# Initialize SimpleImputer with mean strategy
imputer = SimpleImputer(strategy='mean')

# Fit and transform the feature matrix
features_df_imputed = pd.DataFrame(imputer.fit_transform(features_df), columns=features_df.columns)

# Verify that there are no missing values after imputation
missing_values_after = features_df_imputed.isnull().sum()
print(f'Missing values after imputation:\n{missing_values_after}')


Upon inspection, there are no missing values in our feature matrix. However, including the imputation step ensures that our pipeline is robust and can handle datasets with missing values in future analyses.


### Feature Matrix Preparation Summary

The feature matrix is now clean, free from duplicates and missing values, making it ready for dimensionality reduction and further analysis.



In [ ]:
# Final feature matrix
features_df_imputed.head()


# Dimensionality Reduction and Visualization

High-dimensional feature spaces can be challenging to visualize and analyze. Dimensionality reduction techniques like **UMAP** and **t-SNE** help in projecting the data into lower dimensions (typically 2D) while preserving the underlying structure. This projection facilitates visualization and provides insights into the separability of different classes.

### Encoding Labels

Ensure that labels are encoded numerically for compatibility with visualization tools. Label encoding transforms categorical string labels into numerical values, which are essential for machine learning algorithms and visualization techniques.


In [ ]:
# Labels are already encoded earlier using LabelEncoder
# Let's verify the encoding
print("First 10 encoded labels and their original classes:")
print(labels_encoded[:10], labels[:10])


## Step Eleven: Applying UMAP

Uniform Manifold Approximation and Projection (UMAP) is a dimensionality reduction technique that preserves both local and global data structures, making it effective for visualization. It efficiently reduces the high-dimensional feature matrix into a 2D space, preserving the intrinsic data structure.


In [ ]:
# Initialize UMAP reducer with n_jobs=1 to suppress warnings related to random_state
umap_reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)

# Fit and transform the feature matrix
umap_embedding = umap_reducer.fit_transform(features_df_imputed)

# Create a DataFrame for UMAP results
umap_df = pd.DataFrame(umap_embedding, columns=['UMAP1', 'UMAP2'])
umap_df['Label'] = labels
umap_df.head()


## Step Twelve: Applying t-SNE

t-distributed Stochastic Neighbor Embedding (t-SNE) is another popular dimensionality reduction technique. It focuses on preserving local data structures, making it excellent for visualizing clusters within the data. However, it might not capture the global data structure as effectively as UMAP.


In [ ]:
# Initialize t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)

# Fit and transform the feature matrix
tsne_embedding = tsne.fit_transform(features_df_imputed)

# Create a DataFrame for t-SNE results
tsne_df = pd.DataFrame(tsne_embedding, columns=['tSNE1', 'tSNE2'])
tsne_df['Label'] = labels
tsne_df.head()


In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

features_df_imputed_1 = scaler.fit_transform(features_df_imputed)

# Initialize UMAP reducer with n_jobs=1 to suppress warnings related to random_state
umap_reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)

# Fit and transform the feature matrix
umap_embedding = umap_reducer.fit_transform(features_df_imputed_1)

# Create a DataFrame for UMAP results
umap_df = pd.DataFrame(umap_embedding, columns=['UMAP1', 'UMAP2'])
umap_df['Label'] = labels
umap_df.head()

### Visualization of UMAP and t-SNE

Comparing UMAP and t-SNE projections side-by-side provides a comprehensive view of the data's structure and class separability. Observing these projections helps in assessing how well different classes are separated based on the extracted features.

**UMAP** tends to preserve both local and global structures, often resulting in well-separated clusters if the data supports it.

**t-SNE** excels at preserving local neighborhoods, making it ideal for identifying clusters but sometimes distorts global relationships.


In [ ]:
# Define a color palette based on the number of classes
num_classes = len(le.classes_)
palette = sns.color_palette("hsv", num_classes)

# Create a figure with two subplots for UMAP and t-SNE
plt.figure(figsize=(20, 8))

# UMAP Scatter Plot
plt.subplot(1, 2, 1)
sns.scatterplot(
    x='UMAP1', y='UMAP2',
    hue='Label',
    palette=palette,
    data=umap_df,
    legend='full',
    alpha=0.7
)
plt.title('UMAP Projection')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')

# t-SNE Scatter Plot
plt.subplot(1, 2, 2)
sns.scatterplot(
    x='tSNE1', y='tSNE2',
    hue='Label',
    palette=palette,
    data=tsne_df,
    legend='full',
    alpha=0.7
)
plt.title('t-SNE Projection')
plt.xlabel('tSNE1')
plt.ylabel('tSNE2')

# Adjust legend position
plt.legend(title='Class', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
# Define a color palette based on the number of classes
num_classes = len(le.classes_)
palette = sns.color_palette("hsv", num_classes)

# Create a figure with two subplots for UMAP and t-SNE
plt.figure(figsize=(10, 8))

# UMAP Scatter Plot
sns.scatterplot(
    x='UMAP1', y='UMAP2',
    hue='Label',
    palette=palette,
    data=umap_df,
    legend='full',
    alpha=0.7
)
plt.title('UMAP Projection')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')

## Comparison of UMAP and t-SNE
Providing a direct comparison helps understand the strengths and weaknesses of each dimensionality reduction technique.

### Comparative Analysis: UMAP vs. t-SNE

| Feature                        | UMAP                                                   | t-SNE                                                  |
|--------------------------------|--------------------------------------------------------|--------------------------------------------------------|
| **Preservation of Structures** | Preserves both local and global data structures        | Primarily preserves local data structures              |
| **Computation Speed**          | Faster, especially with large datasets                 | Slower, computationally intensive for large datasets   |
| **Scalability**                | Scales well with increasing data size                  | Struggles with very large datasets                      |
| **Parameter Sensitivity**      | Sensitive to parameters like `n_neighbors`             | Less sensitive but requires careful tuning of `perplexity` |
| **Output Consistency**         | More consistent results with fixed `random_state`      | Results can vary between runs due to randomness         |
| **Visualization Quality**      | Provides a balance between cluster separation and structure preservation | Excellent at revealing local clusters but may distort global relationships |
| **Ease of Use**                | Generally easier to use with fewer parameters to tune   | Requires careful parameter tuning for optimal results  |




In [ ]:
# Comparative Analysis: UMAP vs. t-SNE

# UMAP Statistics
umap_stats = pd.DataFrame(umap_embedding, columns=['UMAP1', 'UMAP2'])
print("UMAP Statistics:")
print(umap_stats.describe())

# t-SNE Statistics
tsne_stats = pd.DataFrame(tsne_embedding, columns=['tSNE1', 'tSNE2'])
print("\nt-SNE Statistics:")
print(tsne_stats.describe())


# Classical Machine Learning Methods

Machine learning methods are essential for classifying bioimages based on the extracted features. This section involves implementing classical machine learning algorithms to build predictive models using the feature matrix derived from the Hela dataset. The primary classifiers explored in this step are Decision Trees (DT), K-Nearest Neighbors (KNN), and Support Vector Machines (SVM).




### Split the Feature Matrix and Labels Based on Dataset Splits
Now, we'll split the feature matrix and labels into training, validation, and test sets based on the original dataset splits.

In [ ]:
# Verify that the number of splits matches the number of samples
assert len(split_labels) == features_df_imputed.shape[0], "Mismatch between splits and features."

# Create boolean masks for each split
train_mask = split_labels == 'train'
valid_mask = split_labels == 'valid'
test_mask = split_labels == 'test'

# Split the feature matrix
X_train = features_df_imputed[train_mask]
X_valid = features_df_imputed[valid_mask]
X_test = features_df_imputed[test_mask]

# Split the labels
y_train = labels_encoded[train_mask]
y_valid = labels_encoded[valid_mask]
y_test = labels_encoded[test_mask]

print(f'Training set: {X_train.shape}, Validation set: {X_valid.shape}, Test set: {X_test.shape}')


### Feature Scaling
Scaling features can improve the performance of many machine learning algorithms. We'll use StandardScaler to standardize the feature values.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler on the training data and transform
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test data
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)


### Define Helper Functions for Visualization

To enhance interpretability, we'll define helper functions to plot confusion matrices, ROC curves, and Precision-Recall curves.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, classifier_name, normalize=False, cmap='Blues'):
    """
    Plots a confusion matrix using Seaborn's heatmap.

    Parameters:
    - y_true: Array-like of shape (n_samples,) - True labels.
    - y_pred: Array-like of shape (n_samples,) - Predicted labels.
    - classes: List - Class names.
    - classifier_name: String - Name of the classifier (for the plot title).
    - normalize: Boolean - Whether to apply normalization.
    - cmap: String or Colormap - Color map to use for the heatmap.
    """
    cm = confusion_matrix(y_true, y_pred)

    if normalize:
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
        annot = cm_normalized
    else:
        fmt = 'd'
        annot = cm

    plt.figure(figsize=(10, 8))
    sns.heatmap(annot, annot=True, fmt=fmt, cmap=cmap,
                xticklabels=classes, yticklabels=classes, cbar=True)
    plt.title(f'Confusion Matrix for {classifier_name}', fontsize=16)
    plt.ylabel('True Label', fontsize=14)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.xticks(rotation=45, fontsize=12)
    plt.yticks(rotation=0, fontsize=12)
    plt.tight_layout()
    plt.show()

def plot_roc(y_test_bin, y_score, classifier_name, classes):
    """
    Plots ROC curves for each class.

    Parameters:
    - y_test_bin: Binarized true labels.
    - y_score: Predicted probabilities or decision function scores.
    - classifier_name: String - Name of the classifier.
    - classes: List - Class names.
    """
    n_classes = y_test_bin.shape[1]
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_score[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), y_score.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    plt.figure(figsize=(8, 6))
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f'Class {classes[i]} (AUC = {roc_auc[i]:0.2f})')

    plt.plot([0, 1], [0, 1], 'k--')  # Diagonal line for random guessing
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=14)
    plt.ylabel('True Positive Rate', fontsize=14)
    plt.title(f'Receiver Operating Characteristic - {classifier_name}', fontsize=16)
    plt.legend(loc='lower right', fontsize=12)
    plt.grid(True)
    plt.show()

def plot_precision_recall(y_test_bin, y_score, classifier_name, classes):
    """
    Plots Precision-Recall curves for each class.

    Parameters:
    - y_test_bin: Binarized true labels.
    - y_score: Predicted probabilities or decision function scores.
    - classifier_name: String - Name of the classifier.
    - classes: List - Class names.
    """
    n_classes = y_test_bin.shape[1]
    precision = dict()
    recall = dict()
    average_precision = dict()
    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y_test_bin[:, i], y_score[:, i])
        average_precision[i] = average_precision_score(y_test_bin[:, i], y_score[:, i])

    plt.figure(figsize=(8, 6))
    for i in range(n_classes):
        plt.plot(recall[i], precision[i], label=f'Class {classes[i]} (AP = {average_precision[i]:0.2f})')

    plt.xlabel('Recall', fontsize=14)
    plt.ylabel('Precision', fontsize=14)
    plt.title(f'Precision-Recall Curve - {classifier_name}', fontsize=16)
    plt.legend(loc='upper right', fontsize=12)
    plt.grid(True)
    plt.show()


## Step Thirteen: Model Implementation and Evaluation
In this step, we implement various classical machine learning models, evaluate their performance, and assess how well they classify bioimages using extracted features. To evaluate the performance of the models, several classification metrics are used.

## Decision Trees (DT)
A Decision Tree is a flowchart-like structure where each internal node represents a decision based on the value of a feature, each branch represents the outcome of the decision, and each leaf node represents a class label.



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

# Initialize the Decision Tree classifier with a fixed random state for reproducibility
dt_classifier = DecisionTreeClassifier(random_state=42)

# Train the classifier using the training set
dt_classifier.fit(X_train_scaled, y_train)

# Predict on the validation set to tune hyperparameters (optional)
y_pred_valid_dt = dt_classifier.predict(X_valid_scaled)

# Evaluate the classifier on the validation set
print("Decision Tree Validation Classification Report:")
print(classification_report(y_valid, y_pred_valid_dt))

# Plot Confusion Matrix for Validation Set
plot_confusion_matrix(
    y_true=y_valid,
    y_pred=y_pred_valid_dt,
    classes=le.classes_,
    classifier_name='Decision Tree (Validation)',
    normalize=False
)

# After tuning (if any), predict on the test set
y_pred_test_dt = dt_classifier.predict(X_test_scaled)

# Evaluate the classifier on the test set
print("Decision Tree Test Classification Report:")
print(classification_report(y_test, y_pred_test_dt))

# Plot Confusion Matrix for Test Set
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test_dt,
    classes=le.classes_,
    classifier_name='Decision Tree (Test)',
    normalize=False
)

## K-Nearest Neighbor (KNN)

K-Nearest Neighbors is a simple, instance-based learning algorithm that classifies a new data point based on the majority class among its k nearest neighbors in the feature space.

In [ ]:
# Initialize the KNN classifier with k=5 neighbors
knn_classifier = KNeighborsClassifier(n_neighbors=5)

# Train the classifier using the training set
knn_classifier.fit(X_train_scaled, y_train)

# Predict on the validation set to tune hyperparameters (optional)
y_pred_valid_knn = knn_classifier.predict(X_valid_scaled)

# Evaluate the classifier on the validation set
print("K-Nearest Neighbors Validation Classification Report:")
print(classification_report(y_valid, y_pred_valid_knn))

# Plot Confusion Matrix for Validation Set
plot_confusion_matrix(
    y_true=y_valid,
    y_pred=y_pred_valid_knn,
    classes=le.classes_,
    classifier_name='K-Nearest Neighbors (Validation)',
    normalize=False
)

# After tuning (if any), predict on the test set
y_pred_test_knn = knn_classifier.predict(X_test_scaled)

# Evaluate the classifier on the test set
print("K-Nearest Neighbors Test Classification Report:")
print(classification_report(y_test, y_pred_test_knn))

# Plot Confusion Matrix for Test Set
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test_knn,
    classes=le.classes_,
    classifier_name='K-Nearest Neighbors (Test)',
    normalize=False
)


## Support Vector Machine (SVM)

Support Vector Machines are supervised learning models that analyze data for classification and regression analysis by finding the optimal hyperplane that best separates different classes in the feature space.


#### SVM with Linear Kernel

The linear kernel SVM identifies the best linear boundary between classes, making it suitable for data that are linearly separable. In bioimage informatics, it is effective when dealing with high-dimensional feature spaces, such as basic texture or intensity-based features, where the relationship between classes is straightforward.

In [ ]:
# Initialize the SVM classifier with a linear kernel
svm_classifier = SVC(kernel='linear', probability=True, random_state=42)

# Train the classifier using the training set
svm_classifier.fit(X_train_scaled, y_train)

# Predict on the validation set to tune hyperparameters (optional)
y_pred_valid_svm = svm_classifier.predict(X_valid_scaled)

# Evaluate the classifier on the validation set
print("Support Vector Machine Validation Classification Report:")
print(classification_report(y_valid, y_pred_valid_svm))


# Plot Confusion Matrix for Validation Set
plot_confusion_matrix(
    y_true=y_valid,
    y_pred=y_pred_valid_svm,
    classes=le.classes_,
    classifier_name='SVM (Linear Kernel) (Validation)',
    normalize=False
)

# After tuning (if any), predict on the test set
y_pred_test_svm = svm_classifier.predict(X_test_scaled)

# Evaluate the classifier on the test set
print("Support Vector Machine Test Classification Report:")
print(classification_report(y_test, y_pred_test_svm))


# Plot Confusion Matrix for Test Set
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test_svm,
    classes=le.classes_,
    classifier_name='SVM (Linear Kernel) (Test)',
    normalize=False
)


#### Kernel SVM Example with RBF Kernel:

The RBF kernel SVM maps input data into a higher-dimensional space using a Gaussian function, allowing it to capture complex, non-linear relationships. This is particularly useful in bioimage informatics for classifying images with intricate patterns or subtle variations, such as differentiating between cell phenotypes or identifying subcellular structures where boundaries are not linearly separable.

In [ ]:
# Initialize the SVM classifier with a radial basis function (RBF) kernel
svm_rbf_classifier = SVC(kernel='rbf', probability=True, random_state=42)

# Train the classifier using the training set
svm_rbf_classifier.fit(X_train_scaled, y_train)

# Predict on the validation set to tune hyperparameters (optional)
y_pred_valid_svm_rbf = svm_rbf_classifier.predict(X_valid_scaled)

# Evaluate the classifier on the validation set
print("Support Vector Machine (RBF Kernel) Validation Classification Report:")
print(classification_report(y_valid, y_pred_valid_svm_rbf))

# Plot Confusion Matrix for Validation Set
plot_confusion_matrix(
    y_true=y_valid,
    y_pred=y_pred_valid_svm_rbf,
    classes=le.classes_,
    classifier_name='SVM (RBF Kernel) (Validation)',
    normalize=False
)

# After tuning, predict on the test set
y_pred_test_svm_rbf = svm_rbf_classifier.predict(X_test_scaled)

# Evaluate the classifier on the test set
print("Support Vector Machine (RBF Kernel) Test Classification Report:")
print(classification_report(y_test, y_pred_test_svm_rbf))

# Plot Confusion Matrix for Test Set
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test_svm_rbf,
    classes=le.classes_,
    classifier_name='SVM (RBF Kernel) (Test)',
    normalize=False
)


#### One-vs-Rest SVM

This approach extends linear SVM to multi-class problems by training separate binary classifiers for each class, treating each class against all others. It is often used in bioimage informatics for multi-class classification tasks, such as distinguishing between different cell types or conditions in high-throughput screening, where a simple linear separation suffices between classes.

In [ ]:
# Initialize the One-vs-Rest classifier with SVM using a linear kernel
ovr_svm_classifier = OneVsRestClassifier(SVC(kernel='linear', probability=True, random_state=42))

# Train the classifier using the training set
ovr_svm_classifier.fit(X_train_scaled, y_train)

# Predict on the validation set to tune hyperparameters (optional)
y_pred_valid_ovr_svm = ovr_svm_classifier.predict(X_valid_scaled)

# Evaluate the classifier on the validation set
print("One-vs-Rest SVM Validation Classification Report:")
print(classification_report(y_valid, y_pred_valid_ovr_svm))

# Plot Confusion Matrix for Validation Set
plot_confusion_matrix(
    y_true=y_valid,
    y_pred=y_pred_valid_ovr_svm,
    classes=le.classes_,
    classifier_name='One-vs-Rest SVM (Validation)',
    normalize=False
)

# After tuning (if any), predict on the test set
y_pred_test_ovr_svm = ovr_svm_classifier.predict(X_test_scaled)

# Evaluate the classifier on the test set
print("One-vs-Rest SVM Test Classification Report:")
print(classification_report(y_test, y_pred_test_ovr_svm))

# Plot Confusion Matrix for Test Set
plot_confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test_ovr_svm,
    classes=le.classes_,
    classifier_name='One-vs-Rest SVM (Test)',
    normalize=False
)


## Step Fourteen: N-fold Cross-Validation

Cross-validation provides a more robust estimate of the model's performance by training and evaluating it across multiple subsets of the data.

Process:
* Divide the Dataset: Split the training data into n equal-sized subsets or "folds".
* Training and Validation: For each fold:
* Use n-1 folds for training the model.
* Use the remaining fold for validation.
* Repeat: Perform the training and validation process n times, each time with a different fold as the validation set.
* Average Results: Calculate the average performance metrics across all folds to obtain a reliable estimate of the model's generalization error.

In [ ]:
from sklearn.model_selection import cross_val_score

# Example with Decision Tree classifier
dt_classifier_cv = DecisionTreeClassifier(random_state=42)
cv_scores_dt = cross_val_score(dt_classifier_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
print("Decision Tree Cross-Validation Scores:", cv_scores_dt)
print("Average Cross-Validation Accuracy:", cv_scores_dt.mean())

# Example with K-Nearest Neighbors classifier
knn_classifier_cv = KNeighborsClassifier(n_neighbors=5)
cv_scores_knn = cross_val_score(knn_classifier_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
print("\nK-Nearest Neighbors Cross-Validation Scores:", cv_scores_knn)
print("Average Cross-Validation Accuracy:", cv_scores_knn.mean())

# Example with Support Vector Machine classifier
svm_classifier_cv = SVC(kernel='linear', probability=True, random_state=42)
cv_scores_svm = cross_val_score(svm_classifier_cv, X_train_scaled, y_train, cv=5, scoring='accuracy')
print("\nSupport Vector Machine Cross-Validation Scores:", cv_scores_svm)
print("Average Cross-Validation Accuracy:", cv_scores_svm.mean())


## Step Fifteen: Model Evaluation with ROC and Precision-Recall Curves

While classification metrics offer quantitative insights, visualizing model performance through curves can provide a more intuitive understanding of how classifiers behave under different thresholds.


### Receiver Operating Characteristic (ROC) Curve

A ROC Curve is a graphical plot that illustrates the diagnostic ability of a binary classifier system as its discrimination threshold is varied. It is used to evaluate the trade-off between sensitivity (True Positive Rate) and specificity (1 - False Positive Rate), and to compare different classifiers.


### Precision-Recall Analysis

A Precision-Recall Curve plots the trade-off between Precision and Recall for different threshold settings. It is particularly useful for evaluating models on imbalanced datasets, where the positive class is rare.


#### Binarize the Output Labels for ROC and PR Curves
For multi-class classification, ROC and PR curves need to be computed for each class individually. This involves binarizing the output labels so that each class is treated as the positive class against all other classes.




In [ ]:
# Binarize the output labels for ROC and PR curves
y_test_binarized = label_binarize(y_test, classes=np.arange(len(le.classes_)))
n_classes = y_test_binarized.shape[1]
classes = le.classes_


In [ ]:
# Binarize the output labels for ROC and PR curves
y_test_binarized = label_binarize(y_test, classes=np.arange(len(le.classes_)))
n_classes = y_test_binarized.shape[1]
classes = le.classes_


### Plotting ROC and Precision-Recall Curves

#### Decision Tree Evaluation on Test Set

In [ ]:
# Plot ROC and Precision-Recall Curves for Decision Tree
y_score_dt = dt_classifier.predict_proba(X_test_scaled)
plot_roc(y_test_binarized, y_score_dt, 'Decision Tree', classes)
plot_precision_recall(y_test_binarized, y_score_dt, 'Decision Tree', classes)


#### K-Nearest Neighbors Evaluation on Test Set



In [ ]:
# Plot ROC and Precision-Recall Curves for KNN
y_score_knn = knn_classifier.predict_proba(X_test_scaled)
plot_roc(y_test_binarized, y_score_knn, 'K-Nearest Neighbors', classes)
plot_precision_recall(y_test_binarized, y_score_knn, 'K-Nearest Neighbors', classes)


#### Support Vector Machine Evaluation on Test Set



In [ ]:
# Plot ROC and Precision-Recall Curves for SVM
y_score_svm = svm_classifier.predict_proba(X_test_scaled)
plot_roc(y_test_binarized, y_score_svm, 'Support Vector Machine (Linear Kernel)', classes)
plot_precision_recall(y_test_binarized, y_score_svm, 'Support Vector Machine (Linear Kernel)', classes)


#### SVM with RBF Kernel Evaluation on Test Set



In [ ]:
# Plot ROC and Precision-Recall Curves for SVM with RBF Kernel
y_score_svm_rbf = svm_rbf_classifier.predict_proba(X_test_scaled)
plot_roc(y_test_binarized, y_score_svm_rbf, 'Support Vector Machine (RBF Kernel)', classes)
plot_precision_recall(y_test_binarized, y_score_svm_rbf, 'Support Vector Machine (RBF Kernel)', classes)


#### One-vs-Rest SVM Evaluation on Test Set



In [ ]:
# Plot ROC and Precision-Recall Curves for One-vs-Rest SVM
y_score_ovr_svm = ovr_svm_classifier.predict_proba(X_test_scaled)
plot_roc(y_test_binarized, y_score_ovr_svm, 'One-vs-Rest SVM', classes)
plot_precision_recall(y_test_binarized, y_score_ovr_svm, 'One-vs-Rest SVM', classes)


# Conclusion

This presented a comprehensive workflow for extracting and analyzing texture and geometrical features from bioimages using the Hela dataset, demonstrating a systematic approach to bioimage classification. The workflow encompassed several key steps, including data loading and preprocessing, exploratory data analysis, feature extraction, and the preparation of a feature matrix. Dimensionality reduction and visualization techniques were employed to highlight the structure and relationships within the data, facilitating a deeper understanding of the biological phenomena under investigation.

Classical machine learning methods, including Decision Trees, K-Nearest Neighbors, and Support Vector Machines (SVM) with different kernels, were employed to build predictive models, each providing unique insights into classification performance. The use of ROC and Precision-Recall curves further validated model performance.


# Future Directions

Future enhancements to this bioimage analysis pipeline could include:

- Implementing feature selection techniques such as Recursive Feature Elimination (RFE) or feature importance analysis using models like Random Forests to identify and retain the most relevant features.
- Incorporating image segmentation methods to isolate regions of interest within bioimages, thereby improving the quality and relevance of extracted features.
- Utilizing techniques like SHAP (SHapley Additive exPlanations) or LIME (Local Interpretable Model-agnostic Explanations) to interpret model predictions and understand feature contributions.
- Exploring additional texture and geometrical features to enrich the feature set.